# Содержание

* [Бизнес–контекст (как его формулировали разработчики)](#бизнес–контекст-(как-его-формулировали разработчики))
*

# Бизнес–контекст (как его формулировали разработчики)

Cookie Cats — популярная мобильная puzzle-игра от Tactile Entertainment (жанр «connect three»).

По мере прохождения уровней игрок сталкивается с __гейтами__ — точками, где его останавливают и заставляют либо подождать, либо сделать внутриигровую покупку, чтобы продолжить.

Изначально первый гейт стоял на __30-м уровне__.

Команда провела A/B-тест, в котором первый гейт перенесли на __40-й уровень__, и хотела понять, как это повлияет на удержание игроков.
* Control: gate_30
* Treatment: gate_40

Данные: 90 189 игроков, которые установили игру во время эксперимента.

Каждый игрок случайно попадал в одну из двух групп.

Метрики, которые собирали:
* retention_1 — вернулся ли игрок на следующий день после установки
* retention_7 — вернулся ли на 7-й день
* sum_gamerounds — сколько раундов сыграл за первые 14 дней

In [5]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chisquare, binomtest

In [ ]:
df = pd.read_csv('tables/cookie_cats.csv')
df.sample(5)

# Обработка данных

In [ ]:
df.info()

Пропусков нет.

In [ ]:
df['userid'].duplicated().sum()

Дубликатов по userid нет.

In [ ]:
df[df['sum_gamerounds'] < 0]['sum_gamerounds'].count()

Нет отрицательных чисел в `sum_gamerounds`.

In [ ]:
df['version'].unique()

В колонке `version` только нужные нам значения.

In [ ]:
df[df['version'] == 'gate_30']['sum_gamerounds'].describe()

In [3]:
df[df['version'] == 'gate_40']['sum_gamerounds'].describe()

NameError: name 'df' is not defined

Работа с `sum_gamerounds`

Распределение количества раундов сильно скошено вправо (есть экстремальный выброс 49 854).

Поэтому:
* среднее значение приводим для справки,
* основные выводы и статистические тесты делаем по медиане и с помощью непараметрического критерия (Mann-Whitney U).

Игроков с нулевым количеством раундов не удаляем — это важная часть поведения пользователей.

# Проверка соотношение групп (SRM)

In [ ]:
print(df['version'].value_counts().sort_index())
print(df['version'].value_counts(normalize=True).sort_index()) * 100

Группы распределены не равномерно, посмотрим есть ли srm.

In [ ]:
# chi-square
observed = df['version'].value_counts().sort_index()
expected = [len(df) / 2, len(df) / 2]

chi2, p_chi = chisquare(f_obs=observed, f_exp=expected)

print("Chi-square SRM test")
print(f"Observed: {observed.to_dict()}")
print(f"Expected: {expected}")
print(f"chi2 = {chi2:.3f}, p-value = {p_chi:.4f}")

# binomial test
n_gate40 = (df['version'] == 'gate_40').sum()
n_total = len(df)
binom_result = binomtest(n_gate40, n_total, p=0.5)
print(f"\nBinomial test p-value = {binom_result.pvalue:.4f}")

Для проверки Sample Ratio Mismatch я использую a = 0.05 как базовый порог (стандартный уровень для большинства статистических проверок). При этом даже строгий порог a = 0.01 p-value проходит.

При этом отмечаю, что в индустриальной практике для SRM часто применяют более строгий порог (α = 0.001), потому что это диагностический тест валидности эксперимента, а не тест продуктовой гипотезы.

В нашем случае p-value = 0.0086:
* значим при a = 0.05
* значим при a = 0.01
* не значим при a = 0.001

Поэтому фиксирую наличие лёгкого дисбаланса.

Для проверки Sample Ratio Mismatch (SRM) я использую базовый уровень значимости **α = 0.05**, а также проверяю более строгий порог **α = 0.01**. В индустриальной практике для SRM часто применяют сверхстрогий порог (**α = 0.001**), так как это диагностический тест валидности сплитования, позволяющий отсеять случайные флуктуации на больших выборках и не браковать корректные эксперименты.

В нашем случае получено **p-value = 0.0086** (соотношение групп **49.56% / 50.44%**, разница — 789 пользователей):
* при **α = 0.05** — дисбаланс статистически значим (SRM обнаружен);
* при **α = 0.01** — дисбаланс статистически значим (SRM обнаружен);
* при **α = 0.001** — дисбаланс не является статистически значимым (тест пройден по индустриальному стандарту).

**Вывод и дальнейшие шаги:** Фиксирую наличие лёгкого дисбаланса («пограничная зона»). Поскольку по строгому индустриальному порогу (α = 0.001) тест проходит проверку, а метаданные для поиска технических причин перекоса (дата входа, версия ОС, устройство) в датасете отсутствуют, я продолжаю анализ. Однако все финальные продуктовые выводы будут сопровождаться дисклеймером о возможном смещении выборки.


## Анализ удержания

In [ ]:
grouped = (
    df.groupby('version')
    .agg(
        users=('userid', 'count'),
        sum_gamerounds=('sum_gamerounds', 'sum'),
        retention_1=('retention_1', 'mean'),
        retention_7=('retention_7', 'mean'),
        gamerounds_mean=('sum_gamerounds', 'mean'),
        gamerounds_median=('sum_gamerounds', 'median')
    )
).reset_index()
grouped

In [ ]:
diff_grouped = abs(grouped.iloc[1, 1:] - grouped.iloc[0, 1:])
diff_grouped